# 📝 Notes

## Before Running This Notebook:

1. **Install Dependencies:**
   ```bash
   pip install -r requirements.txt
   ```

2. **Setup Environment Variables:**
   - Copy `.env.example` to `.env`
   - Add your `DEEPSEEK_API_KEY`
   - Configure `WEAVIATE_URL` if needed

3. **Initialize Database:**
   ```bash
   python scripts/setup_database.py
   python scripts/ingest_documents.py
   ```

4. **Verify Weaviate is Running:**
   - Default: http://localhost:8080
   - Check schema is created

## Troubleshooting:

- **ModuleNotFoundError:** Make sure you're running from the notebooks directory
- **Weaviate Connection Error:** Check if Weaviate is running
- **Missing API Key:** Set `DEEPSEEK_API_KEY` in `.env`
- **No Results:** Ensure documents are indexed in Weaviate

In [ ]:
# Cell 11: Quick Query Interface (No Verbose Output)
def quick_query(query: str):
    """
    Quick query with minimal output - just the answer
    
    Args:
        query: User query
    
    Returns:
        Answer string
    """
    print(f"❓ {query}\n")
    response = run_rag_pipeline(query, verbose=False)
    print(f"💡 {response.answer}\n")
    
    if response.citations:
        print(f"📚 Sources: {', '.join(response.citations[:3])}")
    
    return response.answer

# Example quick queries
print("🚀 Quick Query Mode:\n")
print("=" * 80)

quick_query("ควรแปรงฟันอย่างไรหลังผ่าตัด")
print("\n" + "-" * 80 + "\n")

quick_query("ยาแก้ปวดมีผลข้างเคียงอะไรบ้าง")

In [ ]:
# Cell 10: Inspect Document Details
def inspect_document(doc_result):
    """
    Display detailed information about a retrieved document
    
    Args:
        doc_result: SearchResult object
    """
    doc = doc_result.document
    
    print("=" * 80)
    print("📄 DOCUMENT DETAILS")
    print("=" * 80)
    print(f"ID: {doc.id}")
    print(f"Category: {doc.category}")
    print(f"Source File: {doc.source_file}")
    print(f"Page Number: {doc.page_number}")
    print(f"Relevance Score: {doc_result.score:.4f}")
    print(f"Rank: {doc_result.rank}")
    
    print("\n" + "-" * 80)
    print("CONTENT:")
    print("-" * 80)
    print(doc.content)
    
    if doc.metadata:
        print("\n" + "-" * 80)
        print("METADATA:")
        print("-" * 80)
        for key, value in doc.metadata.items():
            print(f"  {key}: {value}")
    
    print("=" * 80)

# Example: Inspect the top document from previous retrieval
if 'retrieval_result' in locals() and retrieval_result.reranked_results:
    print("Inspecting top document from last retrieval:\n")
    inspect_document(retrieval_result.reranked_results[0])
else:
    print("⚠️  No retrieval results available. Run a query first!")

# 🛠️ Utility Functions

## Inspect Retrieved Documents

In [10]:
# Cell 9: Test Category-Specific Retrieval
def test_category_retrieval(query: str, category: str):
    """
    Test retrieval with specific category filter
    
    Args:
        query: User query
        category: Category to filter by
    """
    print(f"🏷️  Testing Category-Filtered Retrieval")
    print("=" * 80)
    print(f"Query: {query}")
    print(f"Category Filter: {category}")
    print("-" * 80)
    
    result = retrieval_service.retrieve_by_category(
        query_text=query,
        category=category,
        top_n=5
    )
    
    print(f"\n✅ Retrieved {len(result.reranked_results)} results from category '{category}'")
    print("\n📄 Results:")
    for i, res in enumerate(result.reranked_results, 1):
        doc = res.document
        preview = doc.content[:100].replace('\n', ' ')
        print(f"\n{i}. [Score: {res.score:.3f}]")
        print(f"   Content: {preview}...")
        print(f"   Category: {doc.category}")
        print(f"   Source: {doc.source_file}")
    
    return result

# Example: Test medication category
category_result = test_category_retrieval(
    query="ควรกินยาแก้ปวดอย่างไร",
    category="Medication"
)

🏷️  Testing Category-Filtered Retrieval
Query: ควรกินยาแก้ปวดอย่างไร
Category Filter: Medication
--------------------------------------------------------------------------------

✅ Retrieved 5 results from category 'Medication'

📄 Results:

1. [Score: 0.615]
   Content: 6 ยาแก้ปวด ยาแก้ปวดมีความส าคัญมากในงานศัลยกรรมช่องปาก ซึ่งโดยปกติจะเป็นการใช้ระงับ ปวดในระดับเล็กน้...
   Category: Medication
   Source: เอกสารประกอบการสอนวิชาคลินิก ภาควิชาศัลยศาสตรช์ช่องปากและแม็กซิลโลเฟเชียล -คณะทันตแพทย์ มช. -Tooth Extraction (หน้า73) -สระและวรรณยุกต์หายหลายตำแหน่ง.pdf

2. [Score: 0.402]
   Content: การรักษาโดยการใช้ยาสามารถท าให้ผู้ปุวยรู้สึกสบายขึ้น การใช้ยายังช่วยให้เกิดการฟื้นฟู สุขภาพของกล้ามเ...
   Category: Medication
   Source: แนวทางการวินิจฉัยและจัดการเบื้องต้นต่อภาวะเจ็บปวดฉุกเฉินด้านทันตกรรมสำหรับนักเรือดำน้ำ -กองวิทยาการ ศูนย์ทันตกรรม กรมแพทย์ทหารเรือ.pdf

3. [Score: 0.303]
   Content: - ยาคลายกล้ามเนื้อ (Muscle relaxants) - ยาต้านภาวะซึมเศร้า ที่ให้ในระดับต่ า (Low-dose antidepressan

In [8]:
# Cell 8: Batch Testing Multiple Queries
def batch_test_queries(queries: list):
    """
    Test multiple queries and compare results
    
    Args:
        queries: List of query strings
    """
    print("🔄 Running Batch Test...")
    print("=" * 80)
    
    results = []
    for i, query in enumerate(queries, 1):
        print(f"\n📍 Query {i}/{len(queries)}: {query}")
        print("-" * 80)
        
        response = run_rag_pipeline(query, verbose=False)
        results.append({
            'query': query,
            'answer': response.answer,
            'citations': len(response.citations),
            'time_ms': response.generation_time_ms
        })
        
        # Show brief result
        answer_preview = response.answer[:150].replace('\n', ' ')
        print(f"✅ Answer: {answer_preview}...")
        print(f"   📝 Citations: {len(response.citations)}")
        print(f"   ⏱️  Time: {response.generation_time_ms:.2f} ms")
    
    print("\n" + "=" * 80)
    print("📊 Batch Test Summary:")
    print("=" * 80)
    avg_time = sum(r['time_ms'] for r in results) / len(results)
    avg_citations = sum(r['citations'] for r in results) / len(results)
    print(f"   Total Queries: {len(results)}")
    print(f"   Avg Time: {avg_time:.2f} ms")
    print(f"   Avg Citations: {avg_citations:.1f}")
    
    return results

# Example batch queries
batch_queries = [
    "หลังผ่าตัดควรทำอะไรบ้าง",
    "อาการบวมหลังผ่าตัดปกติหรือไม่",
    "ต้องงดอาหารประเภทไหนบ้าง"
]

batch_results = batch_test_queries(batch_queries)

🔄 Running Batch Test...

📍 Query 1/3: หลังผ่าตัดควรทำอะไรบ้าง
--------------------------------------------------------------------------------


Top result score 0.36 below threshold 0.5
Insufficient context, returning fallback


✅ Answer: ขออภัยครับ/ค่ะ ไม่มีข้อมูลเพียงพอในระบบ กรุณาปรึกษาทันตแพทย์โดยตรงค่ะ...
   📝 Citations: 0
   ⏱️  Time: 1.49 ms

📍 Query 2/3: อาการบวมหลังผ่าตัดปกติหรือไม่
--------------------------------------------------------------------------------


Context truncated at 4/5 chunks to stay within 4000 chars


✅ Answer: อาการบวมหลังการผ่าตัดในช่องปากเป็นอาการปกติที่สามารถเกิดขึ้นได้ เนื่องจากการผ่าตัดทำให้เกิดการบาดเจ็บของเนื้อเยื่อ นำไปสู่กระบวนการอักเสบของร่างกาย ซึ...
   📝 Citations: 3
   ⏱️  Time: 28436.26 ms

📍 Query 3/3: ต้องงดอาหารประเภทไหนบ้าง
--------------------------------------------------------------------------------


Top result score 0.01 below threshold 0.5
Insufficient context, returning fallback


✅ Answer: ขออภัยครับ/ค่ะ ไม่มีข้อมูลเพียงพอในระบบ กรุณาปรึกษาทันตแพทย์โดยตรงค่ะ...
   📝 Citations: 0
   ⏱️  Time: 2.25 ms

📊 Batch Test Summary:
   Total Queries: 3
   Avg Time: 9480.00 ms
   Avg Citations: 1.0


# 🔬 Advanced Features

## Batch Processing & Category Filtering

In [7]:
# Cell 7: Test Case 3 - Medication Instructions
response3 = run_rag_pipeline("ควรกินยาแก้ปวดตอนไหน")

❓ User Query: ควรกินยาแก้ปวดตอนไหน

🔍 STEP 1: Retrieving Context...
--------------------------------------------------------------------------------


Context truncated at 4/5 chunks to stay within 4000 chars


✅ Retrieved 20 candidates
✅ Reranked to top 5 results
⏱️  Retrieval Time: 624.18 ms

📄 Top Retrieved Documents:
   1. [Score: 0.644] 110 แนวทางปฏิบัติทางทันตกรรมในผู้ป่วยกลุ่มพิเศษ ระยะ 3 เดือนที่ 2 ซึ่งนับจากวันแรกของสัปดาห์ที่ 14 จ...
      Category: Medication | Source: แนวทางเวชปฏิบัติทางทันตกรรม สําหรับคลินิกทันตกรรม-กองทันตสาธารณสุข สํานักอนามัย กรุงเทพมหานคร.pdf
   2. [Score: 0.636] 6 ยาแก้ปวด ยาแก้ปวดมีความส าคัญมากในงานศัลยกรรมช่องปาก ซึ่งโดยปกติจะเป็นการใช้ระงับ ปวดในระดับเล็กน้...
      Category: Medication | Source: เอกสารประกอบการสอนวิชาคลินิก ภาควิชาศัลยศาสตรช์ช่องปากและแม็กซิลโลเฟเชียล -คณะทันตแพทย์ มช. -Tooth Extraction (หน้า73) -สระและวรรณยุกต์หายหลายตำแหน่ง.pdf
   3. [Score: 0.604] การรักษาโดยการใช้ยาสามารถท าให้ผู้ปุวยรู้สึกสบายขึ้น การใช้ยายังช่วยให้เกิดการฟื้นฟู สุขภาพของกล้ามเ...
      Category: Medication | Source: แนวทางการวินิจฉัยและจัดการเบื้องต้นต่อภาวะเจ็บปวดฉุกเฉินด้านทันตกรรมสำหรับนักเรือดำน้ำ -กองวิทยาการ ศูนย์ทันตกรรม กรมแพทย์ทหารเรือ.pdf

🤖 STEP 2: Gene

In [18]:
# Cell 6: Test Case 2 - Nutrition After Surgery
response2 = run_rag_pipeline("หลังผ่าตัดควรกินอาหารอะไร")

❓ User Query: หลังผ่าตัดควรกินอาหารอะไร

🔍 STEP 1: Retrieving Context...
--------------------------------------------------------------------------------


Top result score 0.01 below threshold 0.5
Insufficient context, returning fallback


✅ Retrieved 20 candidates
✅ Reranked to top 5 results
⏱️  Retrieval Time: 627.15 ms

📄 Top Retrieved Documents:
   1. [Score: 0.006] เลื่อดออกหลั่งการผู่าตัด (Secondary hemorrhage) ค้าขาย สุิทธิ์บัตุรทอง อาจเกิ่ดจาก่ก่ารเย็บปัิดแผู้ล...
      Category: Nutrition | Source: การผ่าตัดฟันคุดในฟันกรามซี่ที่สามของขากรรไกรล่าง- การรายงานผู้ป่วยและทบทวนวรรณกรรม -region3 Medical And Dental Journal -Third Molar Surgery.pdf
   2. [Score: 0.005] 7 หากวัสดุแข็งตัวก่อนที่จะปิดไปบนฟัน วัสดุจะไม่ติดกับตัวฟัน ต้องผสมวัสดุใหม่ ควรแนะน าผู้ปุวยไม่ให้ก...
      Category: Nutrition | Source: แนวทางการวินิจฉัยและจัดการเบื้องต้นต่อภาวะเจ็บปวดฉุกเฉินด้านทันตกรรมสำหรับนักเรือดำน้ำ -กองวิทยาการ ศูนย์ทันตกรรม กรมแพทย์ทหารเรือ.pdf
   3. [Score: 0.003] เลื่อดออกหลั่งการผู่าตัด (Secondary hemorrhage) ค้าขายสุิทธิ์บัตุรทองอาจเกิ่ดจาก่ก่ารเย็บปัิดแผู้ลไม...
      Category: Nutrition | Source: การผ่าตัดฟันคุดในฟันกรามซี่ที่สามของขากรรไกรล่าง- การรายงานผู้ป่วยและทบทวนวรรณกรรม -region3 Medical And Dental Journal -Third 

In [12]:
# Cell 5: Test Case 1 - Post-operative Pain
response1 = run_rag_pipeline("ปวดฟันมากหลังผ่าตัด ต้องทำยังไง")

❓ User Query: ปวดฟันมากหลังผ่าตัด ต้องทำยังไง

🔍 STEP 1: Retrieving Context...
--------------------------------------------------------------------------------


Top result score 0.20 below threshold 0.5
Insufficient context, returning fallback


✅ Retrieved 20 candidates
✅ Reranked to top 5 results
⏱️  Retrieval Time: 4828.79 ms

📄 Top Retrieved Documents:
   1. [Score: 0.202] รำยงำนผู้ป่วย (cid:9)Case Report(cid:10) ในช่องปากก่อให้เกิดอาการปวดภายหลังผ่าตัดและการหาย operation...
      Category: Post-op Care | Source: การทำศัลยกรรมผ่าตัดรากฟันออกจากโพรงอากาศแม็กซิลลาและการผ่าตัด ปิดทางเชื่อมระหว่างโพรงอากาศแม็กซิลลาและช่องปาก - กรณีศึกษา-krabi Medical Journal -OAC.pdf
   2. [Score: 0.110] 16 แนวทางปฏิบัติทางทันตกรรมในผู้ป่วยทั่วไป 1 อ 5 แนวทางปฏบิตัใินผปู้ว่ยทมี่คีวามเจบ็ปวดบรเิวณใบหน้าแ...
      Category: Post-op Care | Source: แนวทางเวชปฏิบัติทางทันตกรรม สําหรับคลินิกทันตกรรม-กองทันตสาธารณสุข สํานักอนามัย กรุงเทพมหานคร.pdf
   3. [Score: 0.097] อ7 Chapter 5 Discussion conclusion and suggestion Pain, swelling, and restrict mouth opening are the...
      Category: Post-op Care | Source: ผลของการใช้ยาเดกซาเมทาโซนขนาด 4มก.ภายหลังการผ่าฟันกรามล่างคุด -สถาบันวิจัยมหาวิทยาลัยรังสิต -Third Molar Surgery.pdf

🤖 STEP 2: Generating Respo

# 🧪 Test Cases

Below are example queries to test the RAG system with different types of dental/medical questions.

# 🔬 Test: Retrieval Results & HyDE Output

## Cell สำหรับทดสอบผลลัพธ์จาก Retrieval Pipeline และ HyDE

In [11]:
# Cell: Test Retrieval Results (with reranker scores, chunk content, sources)
def test_retrieval(query: str):
    """
    Test retrieval pipeline and display reranked results.
    
    Input: user query string
    Output: reranked results with score, chunk content, source info
    """
    print("=" * 80)
    print(f"🔍 Query: {query}")
    print("=" * 80)
    
    result = retrieval_service.retrieve(query)
    
    print(f"\n✅ Retrieved {len(result.candidates)} candidates → Reranked to {len(result.reranked_results)} results")
    print(f"⏱️  Retrieval Time: {result.retrieval_time_ms:.2f} ms")
    
    if result.query.hyde_expansion:
        print(f"\n📝 HyDE was used for embedding (see test_hyde_output for details)")
    
    print("\n" + "-" * 80)
    print("📄 Reranked Results:")
    print("-" * 80)
    
    for i, res in enumerate(result.reranked_results, 1):
        doc = res.document
        print(f"\n{'='*60}")
        print(f"📌 Rank {i} | Score: {res.score:.4f}")
        print(f"{'='*60}")
        print(f"📂 Source: {doc.source_file}")
        print(f"📄 Page: {doc.page_number}")
        print(f"🏷️  Category: {doc.category}")
        print(f"\n📖 Content:")
        print(f"{doc.content}")
        print()
    
    return result

# ทดสอบ
query = "หลังผ่าตัดในช่องปาก แนะนำให้รับประทานอาหารอ่อนที่เย็นหรืออุ่นเล็กน้อย เช่น โจ๊ก น้ำซุป หรือไอศกรีม ในช่วง 24-48 ชั่วโมงแรก เพื่อลดการระคายเคืองและเลือดออกค่ะ ควรหลีกเลี่ยงอาหารร้อน อาหารแข็ง หรืออาหารที่ต้องเคี้ยวมากจนกว่าบาดแผลจะหายดีนะคะ"
retrieval_result = test_retrieval(query)

🔍 Query: หลังผ่าตัดในช่องปาก แนะนำให้รับประทานอาหารอ่อนที่เย็นหรืออุ่นเล็กน้อย เช่น โจ๊ก น้ำซุป หรือไอศกรีม ในช่วง 24-48 ชั่วโมงแรก เพื่อลดการระคายเคืองและเลือดออกค่ะ ควรหลีกเลี่ยงอาหารร้อน อาหารแข็ง หรืออาหารที่ต้องเคี้ยวมากจนกว่าบาดแผลจะหายดีนะคะ

✅ Retrieved 20 candidates → Reranked to 5 results
⏱️  Retrieval Time: 5982.55 ms

📝 HyDE was used for embedding (see test_hyde_output for details)

--------------------------------------------------------------------------------
📄 Reranked Results:
--------------------------------------------------------------------------------

📌 Rank 1 | Score: 0.9670
📂 Source: uvulopalatopharyngoplasty-คณะแพทยศาสตร์ศิริราช.pdf
📄 Page: 2
🏷️  Category: Emergency

📖 Content:
ควรหลีกเลยี่งการขากเสมหะแรงๆการลวงคอหรือแปรงฟนเขาไปในชองปากลึกเกินไปการออกแรงมากการเลนกีฬาทหี่กัโหมหรือยกของหนักหลงัผาตัดภายในอ4-48 ชวั่โมงแรกเพราะอาจทำใหมีเลือดออกจากแผลในชองปากไดถามีเลือดออกจากชองปากควรนอนพักยกศีรษะสูงอมน้ำแขง็ในปากนาํ น้ำแข็งหรือ cold pack มาประคบบริเวณหนาผากหรือคอเพ

In [10]:
# Cell: Test HyDE Output (ดูผลลัพธ์จาก HyDE - hypothetical document ที่ deepseek-chat สร้างขึ้น)
def test_hyde_output(query: str):
    """
    Test HyDE generation and display the hypothetical document.
    
    Input: user query string
    Output: hypothetical document generated by deepseek-chat
    """
    print("=" * 80)
    print(f"🔍 Original Query: {query}")
    print("=" * 80)
    
    # Process query (includes HyDE step)
    processed_query = retrieval_service.query_processor.process(query)
    
    print(f"\n📝 Processed Text: {processed_query.processed_text}")
    print(f"🏷️  Category: {processed_query.inferred_category}")
    
    if processed_query.hyde_expansion:
        print(f"\n{'='*80}")
        print("🤖 HyDE Output (Hypothetical Document by deepseek-chat):")
        print(f"{'='*80}")
        print(processed_query.hyde_expansion)
        print(f"\n📏 Length: {len(processed_query.hyde_expansion)} chars")
    else:
        print("\n⚠️  HyDE is not enabled or failed. Check config use_hyde=true and hyde config.")
    
    return processed_query

# ทดสอบ
query = "หลังผ่าตัดควรกินอาหารอะไร"
hyde_result = test_hyde_output(query)

🔍 Original Query: หลังผ่าตัดควรกินอาหารอะไร

📝 Processed Text: หลังผ่าตัดควรกินอาหารอะไร
🏷️  Category: Nutrition

🤖 HyDE Output (Hypothetical Document by deepseek-chat):
หลังผ่าตัดในช่องปาก แนะนำให้รับประทานอาหารอ่อนที่เย็นหรืออุ่นเล็กน้อย เช่น โจ๊ก น้ำซุป หรือไอศกรีม ในช่วง 24-48 ชั่วโมงแรก เพื่อลดการระคายเคืองและเลือดออกค่ะ ควรหลีกเลี่ยงอาหารร้อน อาหารแข็ง หรืออาหารที่ต้องเคี้ยวมากจนกว่าบาดแผลจะหายดีนะคะ

📏 Length: 239 chars


In [1]:
# Cell 1: Setup Path & Autoreload
import sys
import os
from pathlib import Path

# เลื่อน Path ขึ้นไป 1 ชั้นเพื่อให้เจอ folder root project
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# สั่งให้ Notebook โหลด Code ใหม่ทันทีถ้าเราไปแก้ไฟล์ .py (ไม่ต้อง Restart Kernel)
%load_ext autoreload
%autoreload 2

print(f"✅ Project Root: {project_root}")
print(f"✅ Python Path: {sys.path[0]}")

Failed to read module file 'C:\Users\snipe\AppData\Local\Programs\Python\Python311\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "e:\KMITL\Final_PJ\PJ_Rag\env_rag\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\KMITL\Final_PJ\PJ_Rag\env_rag\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\snipe\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1140

✅ Project Root: e:\KMITL\Final_PJ\PJ_Rag
✅ Python Path: e:\KMITL\Final_PJ\PJ_Rag


In [2]:
# Cell 2: Import Required Modules
import warnings
warnings.filterwarnings('ignore')

from config.settings import get_settings
from src.retrieval.service import RetrievalService
from src.generation.engine import GenerationEngine
from src.generation.client import DeepSeekClient
from src.generation.prompt_builder import PromptBuilder
from src.generation.response_parser import ResponseParser

print("✅ All modules imported successfully!")

✅ All modules imported successfully!


In [3]:
# Cell 3: Initialize System
print("🔧 Initializing RAG System...")
print("-" * 60)

# Load settings
settings = get_settings()
print(f"✅ Settings loaded")
print(f"   - Weaviate URL: {settings.weaviate_url}")
print(f"   - DeepSeek API: {'configured' if settings.deepseek_api_key else 'missing'}")

# Initialize Retrieval Service
print("\n🔍 Initializing Retrieval Service...")
try:
    retrieval_service = RetrievalService(settings)
    print(f"✅ Retrieval Service ready")
    print(f"   - Alpha: {retrieval_service.alpha}")
    print(f"   - Top-K: {retrieval_service.top_k}")
    print(f"   - Top-N: {retrieval_service.top_n}")
except Exception as e:
    print(f"❌ Retrieval Service Error: {e}")
    raise

# Initialize Generation Engine
print("\n🤖 Initializing Generation Engine...")
try:
    llm_client = DeepSeekClient(
        api_key=settings.deepseek_api_key,
        model_name=settings.model_config['llm']['model_name'],
        temperature=settings.model_config['llm']['temperature'],
        max_tokens=settings.model_config['llm']['max_tokens']
    )
    
    prompt_builder = PromptBuilder(settings.prompts)
    response_parser = ResponseParser()
    
    generation_engine = GenerationEngine(
        llm_client=llm_client,
        prompt_builder=prompt_builder,
        response_parser=response_parser,
        min_context_score=0.5,
        min_context_count=1
    )
    
    print(f"✅ Generation Engine ready")
    print(f"   - Model: {llm_client.model_name}")
    print(f"   - Temperature: {llm_client.temperature}")
    print(f"   - Max Tokens: {llm_client.max_tokens}")
except Exception as e:
    print(f"❌ Generation Engine Error: {e}")
    raise

print("\n" + "=" * 60)
print("🎉 System Initialized Successfully!")
print("=" * 60)

Failed to read module file 'e:\KMITL\Final_PJ\PJ_Rag\src\retrieval\service.py' for module 'src.retrieval.service': UnicodeDecodeError
Traceback (most recent call last):
  File "e:\KMITL\Final_PJ\PJ_Rag\env_rag\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 219, in update_sources
    self.source_by_modname[new_modname] = f.read()
                                          ^^^^^^^^
  File "C:\Users\snipe\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 1701: character maps to <undefined>
Failed to read module file 'e:\KMITL\Final_PJ\PJ_Rag\src\retrieval\query_processor.py' for module 'src.retrieval.query_processor': UnicodeDecodeError
Traceback (most recent call last):
  File "e:\KMITL\Final_PJ\PJ_Rag\env_rag\Lib\site-pac

🔧 Initializing RAG System...
------------------------------------------------------------
✅ Settings loaded
   - Weaviate URL: http://localhost:8080
   - DeepSeek API: configured

🔍 Initializing Retrieval Service...


e:\KMITL\Final_PJ\PJ_Rag\env_rag\Lib\site-packages\weaviate\warnings.py:133: DeprecationWarning: Dep005: You are using weaviate-client version 4.7.1. The latest version is 4.19.2.
            Consider upgrading to the latest version. See https://weaviate.io/developers/weaviate/client-libraries/python for details.
  warnings.warn(
e:\KMITL\Final_PJ\PJ_Rag\env_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\KMITL\Final_PJ\PJ_Rag\env_rag\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Fetching 30 files: 100%|██████████| 30/30 [00:00<?, ?it/s]


✅ Retrieval Service ready
   - Alpha: 0.7
   - Top-K: 20
   - Top-N: 5

🤖 Initializing Generation Engine...
✅ Generation Engine ready
   - Model: deepseek-reasoner
   - Temperature: 0.1
   - Max Tokens: 16384

🎉 System Initialized Successfully!


In [4]:
# Cell 4: Define RAG Pipeline Function
def run_rag_pipeline(query_text: str, verbose: bool = True):
    """
    Run complete RAG pipeline: Retrieval → Generation → Display Results
    
    Args:
        query_text: User query in Thai
        verbose: Whether to print detailed logs
    
    Returns:
        GeneratedResponse object
    """
    if verbose:
        print("=" * 80)
        print(f"❓ User Query: {query_text}")
        print("=" * 80)
    
    # Step 1: Retrieve relevant documents
    if verbose:
        print("\n🔍 STEP 1: Retrieving Context...")
        print("-" * 80)
    
    retrieval_result = retrieval_service.retrieve(query_text)
    
    if verbose:
        print(f"✅ Retrieved {len(retrieval_result.candidates)} candidates")
        print(f"✅ Reranked to top {len(retrieval_result.reranked_results)} results")
        print(f"⏱️  Retrieval Time: {retrieval_result.retrieval_time_ms:.2f} ms")
        
        # Show top results
        print("\n📄 Top Retrieved Documents:")
        for i, result in enumerate(retrieval_result.reranked_results[:3], 1):
            doc = result.document
            preview = doc.content[:100].replace('\n', ' ')
            print(f"   {i}. [Score: {result.score:.3f}] {preview}...")
            print(f"      Category: {doc.category} | Source: {doc.source_file}")
    
    # Step 2: Generate response
    if verbose:
        print("\n🤖 STEP 2: Generating Response...")
        print("-" * 80)
    
    response = generation_engine.generate(retrieval_result, query_text)
    
    if verbose:
        print(f"✅ Response generated")
        print(f"⏱️  Generation Time: {response.generation_time_ms:.2f} ms")
    
    # Step 3: Display results
    if verbose:
        print("\n" + "=" * 80)
        print("💡 ANSWER:")
        print("=" * 80)
        print(response.answer)
        
        print("\n" + "-" * 80)
        print("📚 CITATIONS:")
        print("-" * 80)
        if response.citations:
            for i, cite in enumerate(response.citations, 1):
                print(f"   [{i}] {cite}")
        else:
            print("   (No citations)")
        
        # Summary
        total_time = retrieval_result.retrieval_time_ms + response.generation_time_ms
        print("\n" + "=" * 80)
        print("📊 SUMMARY:")
        print("=" * 80)
        print(f"   ⏱️  Total Time: {total_time:.2f} ms")
        print(f"   📄 Documents Used: {len(response.context_used)}")
        print(f"   📝 Citations: {len(response.citations)}")
        print(f"   🤖 Model: {response.model_name}")
        print("=" * 80)
    
    return response

print("✅ RAG Pipeline function defined successfully!")

✅ RAG Pipeline function defined successfully!
